In [ ]:
import math
import os
import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [ ]:
torch.manual_seed(1337)

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int

    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 256
    d_ff: int = 4 * 256
    dropout: float = 0.1

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, causal=True):
        """
        x: (B,T,D)
        returns: (B,T,D)
        """
        B, T, D = x.shape

        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1,2)

        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.d_head)

        if causal:
            mask = torch.triu(
                torch.ones(T, T, device = x.device, dtype = torch.bool),
                diagonal = 1
            )
            scores = scores.masked_fill(mask.view(1, 1, T, T), float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.resid_drop(self.wo(out))
        return out

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        x: (B,T,D)
        returns: (B,T,D)
        """
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = MultiHeadAttention(config.d_model, config.n_heads, config.dropout)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ff = FeedForward(config.d_model, config.d_ff, config.dropout)

    def forward(self, x):
        """
        x: (B,T,D)
        returns: (B,T,D)
        """
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.block_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(config) for _ in range(config.n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        
        # Weight tying
        self.token_embedding.weight = self.lm_head.weight
        
        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        """
        idx: (B, T) tensor of token indices
        targets: (B, T) tensor of target token indices (optional)
        returns: logits (B, T, vocab_size) and optionally loss
        """
        B, T = idx.shape
        assert T <= self.config.block_size, f"Sequence length {T} exceeds block size {self.config.block_size}"
        
        # Token and position embeddings
        tok_emb = self.token_embedding(idx)  # (B, T, D)
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)  # (T,)
        pos_emb = self.position_embedding(pos)  # (T, D)
        
        x = self.dropout(tok_emb + pos_emb)  # (B, T, D)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.ln_f(x)  # (B, T, D)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate new tokens from a conditioning sequence.
        idx: (B, T) tensor of conditioning indices
        max_new_tokens: number of tokens to generate
        temperature: softmax temperature (higher = more random)
        top_k: if set, only sample from top k most likely tokens
        """
        for _ in range(max_new_tokens):
            # Crop context if needed
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            
            # Forward pass
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # (B, vocab_size)
            
            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            
            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            # Append to sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        
        return idx

In [ ]:
# Example: Character-level text dataset
class CharDataset:
    def __init__(self, text, block_size):
        self.block_size = block_size
        
        # Build vocabulary
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        
        # Encode entire text
        self.data = torch.tensor([self.stoi[ch] for ch in text], dtype=torch.long)
    
    def encode(self, text):
        return [self.stoi[ch] for ch in text]
    
    def decode(self, indices):
        return ''.join([self.itos[i] for i in indices])
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def get_batch(self, batch_size):
        """Get a random batch of data"""
        ix = torch.randint(len(self) - 1, (batch_size,))
        x = torch.stack([self.data[i:i+self.block_size] for i in ix])
        y = torch.stack([self.data[i+1:i+self.block_size+1] for i in ix])
        return x, y

In [ ]:
# Load sample text (Shakespeare)
try:
    with open('input.txt', 'r', encoding='utf-8') as f:
        text = f.read()
except FileNotFoundError:
    # If no file, use sample text
    text = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them."""
    print("Using sample text (input.txt not found)")

print(f"Text length: {len(text):,} characters")

# Create dataset
block_size = 128
dataset = CharDataset(text, block_size)
print(f"Vocabulary size: {dataset.vocab_size}")
print(f"Dataset size: {len(dataset):,} examples")

In [ ]:
# Initialize model
config = GPTConfig(
    vocab_size=dataset.vocab_size,
    block_size=block_size,
    n_layers=4,
    n_heads=4,
    d_model=256,
    d_ff=1024,
    dropout=0.1
)

model = GPT(config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)

In [ ]:
# Training loop
batch_size = 32
num_steps = 5000
eval_interval = 500

model.train()
for step in range(num_steps):
    # Get batch
    x, y = dataset.get_batch(batch_size)
    x, y = x.to(device), y.to(device)
    
    # Forward pass
    logits, loss = model(x, y)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    # Logging
    if step % eval_interval == 0 or step == num_steps - 1:
        print(f"Step {step:5d} | Loss: {loss.item():.4f}")

print("\nTraining complete!")

In [ ]:
# Generate some text
model.eval()

# Start with a prompt
prompt = "To be"
context = torch.tensor([dataset.encode(prompt)], dtype=torch.long, device=device)

# Generate
generated = model.generate(context, max_new_tokens=200, temperature=0.8, top_k=40)
generated_text = dataset.decode(generated[0].tolist())

print("=" * 50)
print("Generated text:")
print("=" * 50)
print(generated_text)